In [0]:
-- ============================================================
-- KRAF | FASE 2 | CONSULTA OLAP 01
-- Del Registro al Contexto
-- Responsable: Karen María Mina Martínez
--
-- Pregunta de negocio:
-- ¿Existen comportamientos transaccionales inusuales asociados
-- a variaciones de saldo y al contexto económico?
--
-- Objetivo:
-- Relacionar las transferencias de KRAF, las variaciones de
-- saldo y el contexto económico disponible de El Salvador para
-- identificar periodos con mayor actividad transaccional y
-- posibles señales atípicas.
--
-- Nota:
-- El dataset económico disponible para El Salvador llega hasta
-- 2023. Por ello, se utiliza 2023 como último contexto económico
-- disponible para complementar el análisis de 2025.
-- ============================================================

WITH transferencias_mensuales AS (

    SELECT
        YEAR(fecha_hora) AS anio,
        MONTH(fecha_hora) AS mes,
        COUNT(*) AS total_transferencias,
        SUM(monto) AS monto_total,
        SUM(
            CASE
                WHEN clasificacion = 'ATIPICA' THEN 1
                ELSE 0
            END
        ) AS transferencias_atipicas,
        AVG(monto) AS monto_promedio

    FROM workspace.kraf_gold.transferencias_riesgo

    GROUP BY
        YEAR(fecha_hora),
        MONTH(fecha_hora)
),

saldos_mensuales AS (

    SELECT
        YEAR(fecha) AS anio,
        MONTH(fecha) AS mes,
        SUM(variacion_saldo) AS variacion_total_saldos,
        AVG(variacion_saldo) AS variacion_promedio_saldo

    FROM workspace.kraf_silver.saldo_historico_silver

    GROUP BY
        YEAR(fecha),
        MONTH(fecha)
),

contexto_economico AS (

    SELECT
        year AS anio_contexto,
        inflation_rate,
        rating_foreign,
        rating_foreign_outlook

    FROM workspace.kraf_gold.contexto_economico_pais

    WHERE country_code = 'SLV'
      AND year = 2023
)

SELECT
    t.anio,
    t.mes,
    t.total_transferencias,
    ROUND(t.monto_total, 2) AS monto_total,
    t.transferencias_atipicas,
    ROUND(t.monto_promedio, 2) AS monto_promedio,
    ROUND(s.variacion_total_saldos, 2) AS variacion_total_saldos,
    ROUND(s.variacion_promedio_saldo, 2) AS variacion_promedio_saldo,

    -- Contexto económico disponible
    e.anio_contexto,
    ROUND(e.inflation_rate, 2) AS inflation_rate,
    e.rating_foreign,
    e.rating_foreign_outlook

FROM transferencias_mensuales t

LEFT JOIN saldos_mensuales s
    ON t.anio = s.anio
    AND t.mes = s.mes

CROSS JOIN contexto_economico e

ORDER BY
    t.anio,
    t.mes;